# Comparative Reliability Performance of N-Body Simulation under Hamiltonian Framework Applying Barnes-Hut Trees
Within the realms of modern orbital mechanics application of Barnes-Hut trees is a common practice for large N-Body simulations. The project describes the creation of a C++ and R tool to simulate the problem assessing reliability of the Barnes-Hut trees for cosmological applications via application of symplectic and non-symplectic integration schemes.

## About The Notebook
This notebook is a simulation prototype, that shows fast results and validation of said results using Python.
It will be kept in a dated entry order expanding each individual feature for a later transfer to C++ and R framework.

Please bare in mind that the function and variable names follows a certain pattern:
- Integrators and their daughter functions: `operation_integrator_order`
- Mathematical equations take their literal mathematical names, where numerator is a numerator, product is a product, etc.

## Setting the Environment and Creating Velocity Verlet 4th Order (28/10/26)

In [ ]:
# Imports
import matplotlib as mp
import numpy as np
import math
import random as rnd

# Randomisation Seed
rnd.seed(256)

# Constants
G_const = 6.67e-11
yoshida_x0 = (2**(1/3)/(2-(2**1/3)))
yoshida_x1 = (1/(2-(2**(1/3))))

In [ ]:
"""
Creating helper functions to help with force evaluation later.
"""

# Find Euclidian distance between two bodies
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
def euclidian_distance(obj_a, obj_b):
    return np.sqrt(obj_a.r**2 + obj_b.r**2)

# Find initial velocity
# obj – individual object with accessible properties
# G – constant of gravitational attraction
# distance – previously found Euclidian distance
def vel_init(obj, G, distance):
    return rnd.random() * np.sqrt((G*obj.m) / distance)  # random variable in the beginning is used to create random orbit eccentricities (no ejection)

# Get a perpendicular vector
def get_perpendicular(v):
    x, y, z = v
    if x != 0 or y != 0:
        perp = np.array([-y, x, 0.0], dtype=float)
    else:
        perp = np.array([0.0, 1.0, 0.0], dtype=float)

    return perp

# Find velocity direction
# Essentially convert scalar velocity into vector velocity
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
# G – constant of gravitational attraction
# distance – previously found Euclidian distance
# v – scalar velocity
def vel_direction(obj_a, obj_b, distance, v):
    radius_vector = obj_b - obj_a
    unit_perp_vector = get_perpendicular(radius_vector) / distance

    return v * unit_perp_vector


# Compute Gravitational Acceleration of All Bodies acting on i
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# e – softening factor to prevent division by 0 errors in case case
#     the bodies get too close.
def acc(G, bod_obj, e):
    N = len(bod_obj)    
    accelerations = [np.zeros(3) for _ in range(N)]     # initialise an array to hold acceleration vectors for all bodies

    # Sigma operation
    # Loop over every body i
    for i in range(N):
        a_i = np.zeros(3)   # net acceleration vector on body i

        # Sum gravitational pull from every other body j
        for j in range(N):
            if i == j:
                continue    # idiot-proofing self-interaction

            # Compute displacement from i to j
            r_ij = bod_obj[i].r - bod_obj[j].r

            # Scalar distance squared
            dist_sq = np.sum(r_ij**2)

            # Denominator with softening factor
            denominator = (dist_sq + e**2)**1.5

            # Add the j-body contribution
            a_i += bod_obj[j].m * (r_ij / denominator)

        # Apply -G and store the net acceleration for body i
        accelerations[i] = -G * a_i

    return accelerations


In [ ]:
"""
Creating a Velocity Verlet 4th Order Script.
It consists of the acceleration, displacement and velocity computations.

The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of a an individual body
# obj – individual object with accessible properties
# t – current timestep
def displ_vv_4(obj, t):
    return obj.r + obj.v*t + (0.5*obj.a*(t**2))

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# a_old – old acceleration of the body
def vel_vv_4(obj, t, a_old):
    return obj.v + 0.5*(a_old+obj.a)*t

# Run Velocity Verlet
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# e – softening factor to prevent division by 0 errors in case case
#     the bodies get too close.
def run_vv_4(bod_obj, G, e, dt):
    # Start the clock of the simulated Universe
    for t in range(dt):
        # Run through all the bodies
            for i in range(len(bod_obj)):   # i is the protagonist body
                # Sum gravitational pull from every other body j
                for j in range((len(bod_obj))):
                    if i == j:
                        continue    # idiot-proofing self-interaction
        
                    # Set initial conditions
                    dist = euclidian_distance(bod_obj[i], bod_obj[j])   # find Euclidian distance
                    speed = vel_init(bod_obj[i], G, dist)   # find initial speed
                    velocity = vel_direction(bod_obj[i], bod_obj[j], dist, speed)   # find velocity

                    # Integrate (Step 1)
                    a_init = bod_obj[i].a = acc(G, bod_obj, e)[i]     # list of initial accelerations with assignments of the initial acceleration

                    bod_obj[i].r = displ_vv_4(bod_obj[i], t * yoshida_x1)    # new displacement
                    dist = euclidian_distance(bod_obj[i], bod_obj[j])   # new distance between the objects

                    bod_obj[i].a = acc(G, bod_obj, e)[i]    # new acceleration

                    bod_obj[i].v = vel_vv_4(bod_obj[i], t * yoshida_x1, a_init)  # new velocity

                    # Integrate (Step 2)
                    a_init = bod_obj[i].a = acc(G, bod_obj, e)[i]     # list of initial accelerations with assignments of the initial acceleration
                    
                    bod_obj[i].r = displ_vv_4(bod_obj[i], t * yoshida_x0)    # new displacement
                    dist = euclidian_distance(bod_obj[i], bod_obj[j])   # new distance between the objects
                    
                    bod_obj[i].a = acc(G, bod_obj, e)[i]    # new acceleration
                    
                    bod_obj[i].v = vel_vv_4(bod_obj[i], t * yoshida_x0, a_init)  # new velocity

                    # Integrate (Step 3)
                    a_init = bod_obj[i].a = acc(G, bod_obj, e)[i]     # list of initial accelerations with assignments of the initial acceleration
                                        
                    bod_obj[i].r = displ_vv_4(bod_obj[i], t * yoshida_x0)    # new displacement
                    dist = euclidian_distance(bod_obj[i], bod_obj[j])   # new distance between the objects
                                        
                    bod_obj[i].a = acc(G, bod_obj, e)[i]    # new acceleration
                                        
                    bod_obj[i].v = vel_vv_4(bod_obj[i], t * yoshida_x0, a_init)  # new velocity